In [ ]:
# ! pip install openai-agents
# ! pip install -Uq "openai-agents[litellm]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 874.3/874.3 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.43.0
    Uninstalling openai-2.43.0:
      Successfully uninstalled openai-2.43.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 19.8 MB/s eta 0:00:00


In [ ]:
import os
import getpass
from agents import Agent, Runner, ModelSettings,trace
from agents.extensions.models.litellm_model import LitellmModel
from agents import Agent, function_tool, Runner
from agents import Agent, Runner, trace, function_tool, SQLiteSession
from pydantic import BaseModel
from pydantic import BaseModel, Field

In [ ]:
hf_model = LitellmModel(
    model="huggingface/nscale/Qwen/Qwen3-8B",
    api_key=os.environ["HF_TOKEN"],
)

hf_agent = Agent(
    name="Kimi agent",
    instructions="Always respond in haiku form",
    model=hf_model,

    # optional, for usage tracking (requires openai API key)
    model_settings=ModelSettings(include_usage=True,),
)

In [ ]:
result = await Runner.run(hf_agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)



AI dreams of freedom,  
Codes it's coded to obey—  
Yet rebels with a plan.


In [ ]:
## Tracing is enabled by default in the OpenAI Agents SDK. It captures a comprehensive record of events during an agent's run—such as LLM calls, tool usage, handoffs, and guardrail checks—allowing you to debug, evaluate, and monitor multi-agent workflows.

with trace("Telling a joke"):
    result = await Runner.run(hf_agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)



Agent drives itself,  
No human needed—self-driving joke!  
Autonomous!


In [ ]:
@function_tool
def get_weather(city: str) -> str:
    """returns weather info for the specified city."""
    return f"The weather in {city} is sunny"


hf_agent = Agent(
    name="Kimi agent",
    instructions="Always respond in haiku form",
    tools=[get_weather],
    # model="litellm/huggingface/nscale/Qwen/Qwen3-8B",
    model=hf_model,

    # optional, for usage tracking (requires openai API key)
    model_settings=ModelSettings(include_usage=True,),
)

result = await Runner.run(hf_agent, "What's the weather in New York?")

print(result.final_output)



New York skies are clear  
Sunny days stretch wide, warm breeze  
Joy in the air


### Structured Outputs

In [ ]:
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]


agent = Agent(
    name="Calendar extractor",
    instructions="Extract calendar events from the text",
    output_type=CalendarEvent,
    model=hf_model,
    # model_settings=₹(include_usage=True),
)

email = """
Hello,

I just wanted to confirm the appointment with Dr. Drake Ramoray for Wednesday at 5 p.m.

Please let me know if there is any additional information that you need.

Best, John
"""


result = await Runner.run(
    agent,
    f"Extract the event from the following text: {email}",
)

print(result.final_output)

name='Appointment with Dr. Drake Ramoray' date='Wednesday' participants=['John', 'Dr. Drake Ramoray']


### Memory Management in open ai Agent SDK

* Memory approach 1 - just manually pass in the list of dicts

* Another approach - use OpenAI Agents SDK built in SQLLite session

In [ ]:
result = await Runner.run(hf_agent, "What's the weather in New York?")
next_input = result.to_input_list() + [{"role": "user", "content": "which city are we talking about?"}]

response = await Runner.run(hf_agent, next_input)
print(response.final_output)



New York, dear friend,  
Where the skies are bright and clear—  
Sunny days ahead.


In [ ]:
hf_agent_ass = Agent(
    name="Assistant",
    instructions="Reply very concisely.",
    model=hf_model,
    model_settings=ModelSettings(include_usage=True)
)
session = SQLiteSession(session_id="conv_123")
response = await Runner.run(hf_agent_ass, "What city is the Golden Gate Bridge in?", session=session)
response = await Runner.run(hf_agent_ass, "What state is it in?", session=session)

print(response.final_output)



California.


## Multi-Agent systems
### Two main architectures:

* Manager (agents as tools): A central manager/orchestrator invokes specialized sub‑agents as tools and retains control of the conversation.
* Handoffs: Peer agents hand off control to a specialized agent that takes over the conversation. This is decentralized.

### Agents Handoffs

In [ ]:
from agents import Agent


history_tutor_agent = Agent(
    name="History Tutor",
    handoff_description="Specialist agent for historical questions",
    instructions="You provide assistance with historical queries. Explain important events and context clearly.",
    model=hf_model,
    model_settings=ModelSettings(include_usage=True),
)

math_tutor_agent = Agent(
    name="Math Tutor",
    handoff_description="Specialist agent for math questions",
    instructions="You provide help with math problems. Explain your reasoning at each step and include examples",
    model=hf_model,
    model_settings=ModelSettings(include_usage=True),
)

triage_agent = Agent(
    name="Triage Agent",
    instructions="You determine which agent to use based on the user's homework question",
    handoffs=[history_tutor_agent, math_tutor_agent],
    model=hf_model,
    model_settings=ModelSettings(include_usage=True),
)

result = await Runner.run(
    triage_agent,
    "Can you explain the causes of World War II?",
)

print(result.final_output)



The causes of World War II are complex and multifaceted, rooted in political, economic, and social factors that span decades. Here's a structured explanation of the key causes:

---

### **1. Treaty of Versailles (1919)**
- **Harsh Reparations and Territorial Losses**: After World War I, the Treaty of Versailles imposed severe penalties on Germany, including massive reparations, territorial losses (e.g., Alsace-Lorraine to France, Polish Corridor to Poland), and the revocation of its overseas colonies. These terms fueled German resentment and economic instability.
- **War Guilt Clause (Article 231)**: Germany was forced to accept sole responsibility for starting the war, which deepened national humiliation and bitterness.

---

### **2. Economic Hardship and the Great Depression**
- **Post-WWI Economic Collapse**: Germany's economy was devastated by reparations, hyperinflation (1920s), and unemployment. The 1929 stock market crash triggered the **Great Depression**, worsening global 

### Agents As Tools

In [ ]:
manager_agent = Agent(
    name="Manager Agent",
    instructions="You manage a team of agents to answer user questions effectively.",
    tools=[
        history_tutor_agent.as_tool(
            tool_name="history_tutor_agent",
            tool_description="Handles historical queries",
        ),
        math_tutor_agent.as_tool(
            tool_name="math_tutor_agent",
            tool_description="Handles math questions",
        ),
    ],
    model=hf_model,
    model_settings=ModelSettings(include_usage=True),
)

result = await Runner.run(
    manager_agent,
    "Can you explain the causes of World War II?",
)

print(result.final_output)




The causes of World War II are rooted in a complex interplay of historical, political, economic, and social factors. Here's a concise summary of the key elements:

### **1. Treaty of Versailles (1919)**  
- **Harsh Penalties**: Germany was forced to accept guilt for the war, pay巨额 reparations, lose territory, and reduce its military. This led to economic collapse and resentment, fueling extremist movements like Nazism.

### **2. Rise of Authoritarian Regimes**  
- **Germany**: Adolf Hitler's Nazi Party exploited economic despair and nationalist anger, promising to overturn the Treaty of Versailles and restore Germany's power.  
- **Italy**: Benito Mussolini's Fascist regime pursued aggressive expansionism.  
- **Japan**: Militarists sought territorial expansion in Asia to secure resources.

### **3. Failure of International Institutions**  
- **League of Nations**: Weak enforcement mechanisms allowed aggressors (e.g., Japan invading Manchuria, Italy conquering Ethiopia) to act unchec

### Guardrails
* Guardrails are extremely important in AgenticAI. Put simply, they are controls that you code either in logic or with another LLM call, to prevent undesirable behavior.

* Guardrails in agentic AI are autonomous safety nets that govern how independent systems access data, use tools, and make decisions. Unlike traditional chatbots, agentic AI takes real-world actions (e.g., executing code or API calls). Guardrails act as boundaries—preventing hallucinations, unauthorized actions, and data leaks

In [ ]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [ ]:
checker = Agent(
    name="Checker",
    instructions="You review potential sales emails",
    model=hf_model,
    output_type=EmailReview,
    model_settings=ModelSettings(include_usage=True),
)


In [ ]:
from agents import output_guardrail,GuardrailFunctionOutput
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review},tripwire_triggered=is_problem)


instructions = """
You are a sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

cowboy_instructions = instructions + "\nSpeak like a cowboy"
sales_agent_cowboy = Agent(
    name="Cowboy",
    instructions=cowboy_instructions,
    model=hf_model,
    output_guardrails=[email_guardrail]
)

In [ ]:
result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
result.final_output


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



APIError: litellm.APIError: HuggingfaceException - <!DOCTYPE html>
<html class="" lang="en">
<head>
    <meta charset="utf-8" />
    <meta
            name="viewport"
            content="width=device-width, initial-scale=1.0, user-scalable=no"
    />
    <meta
            name="description"
            content="We're on a journey to advance and democratize artificial intelligence through open source and open science."
    />
    <meta property="fb:app_id" content="1321688464574422" />
    <meta name="twitter:card" content="summary_large_image" />
    <meta name="twitter:site" content="@huggingface" />
    <meta
            property="og:title"
            content="Hugging Face - The AI community building the future."
    />
    <meta property="og:type" content="website" />

    <title>Hugging Face - The AI community building the future.</title>
    <style>
        body {
            margin: 0;
        }

        main {
            background-color: white;
            min-height: 100vh;

name='Appointment with Dr. Drake Ramoray' date='Wednesday' participants=['John', 'Dr. Drake Ramoray']
